In [1]:
import pandas as pd
import numpy as np

## Explanation

This file takes all satellite images from locations in the lake dataset for the 90 days previous to each sample and transforms it into four csvs that will be used for model training. They are structured to be able to load into an LSTM model and are designed to train four different forecast models --> 1 week in advance, 2 weeks, 3 weeks, and 1 month. 

In [2]:
pastdf = pd.read_csv('/Users/emersonfrost/Desktop/caml_90days_previous.csv')
print(pastdf.shape)

(194235, 18)


In [4]:
#group to get unique 
counts = (
    pastdf
    .groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
    .size()
    .reset_index(name='n_obs')
)
print('general statistics')
print(np.mean(counts.cyanobacteria_abundance))
print(np.std(counts.cyanobacteria_abundance))
print(np.median(counts.cyanobacteria_abundance))
print(min(counts.cyanobacteria_abundance))
print(max(counts.cyanobacteria_abundance))

#looking at these stats for the possibility of a classifier in the future
print('above 20_000')
print(counts[counts['cyanobacteria_abundance'] >= 20000].shape[0])
print('below')
print(counts[counts['cyanobacteria_abundance'] < 20000].shape[0])
print('above 100_000')
print(counts[counts['cyanobacteria_abundance'] >= 100000].shape[0])
print('below')
print(counts[counts['cyanobacteria_abundance'] < 100000].shape[0])

general statistics
1915.1198420016817
3882.4973475969787
225.77099609375
0.0
154213.8796
above 20_000
24
below
7928
above 100_000
2
below
7950


In [7]:
#general cleaning
print('before removing nulls', pastdf.shape)
pastdf = pastdf[(pastdf['green'] != -999) | (pastdf['blue'] != -999) | (pastdf['SWIR'] != -999) | (pastdf['NIR'] != -999)]
print('removed nulls', pastdf.shape)
#removing null values / those that didn't populate
#under 70% cloud coverage (want to remove extreme cloud coverage)
print('before cloudiness change', pastdf.shape)
pastdf = pastdf[pastdf['cloud_score'] <= 0.7]
print("Under 70%: ", pastdf.shape)

#drop duplicate entries & take one with less cloud coverage
pastdf = (pastdf.sort_values('cloud_score', ascending = True).drop_duplicates(subset = ['cyanobacteria_abundance','sample_date', 'satellite_date']))
print('removed duplicates:', pastdf.shape)

before removing nulls (106343, 18)
removed nulls (106343, 18)
before cloudiness change (106343, 18)
Under 70%:  (106343, 18)
removed duplicates: (106343, 18)


In [9]:
#to determine number of time series points --> see how many possible values for each time variable
#list of number of days we need to drop before
pastdf["sample_date"] = pd.to_datetime(pastdf["sample_date"])
pastdf["satellite_date"] = pd.to_datetime(pastdf["satellite_date"])
# 1 week, 2 week, 3 week, 1 month
days_list = [7, 14, 21, 30]
for i in range(len(days_list)):
    print(f"Possible points for {days_list[i]} days")
    #return a subset of data without x days within sample date
    filtered_df = pastdf[
        (pastdf["satellite_date"] - pastdf["sample_date"]).abs().dt.days >= days_list[i]
    ]

    # then group
    counts = (
        filtered_df
        .groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
        .size()
        .reset_index(name='n_obs')
    )

    total = len(counts)
    print("total: ", total)
    
    #check 5, 6, 7, 8, 9 number of time series points 
    n_5 = (counts['n_obs'] >= 5).sum()
    n_6 = (counts['n_obs'] >= 6).sum()
    n_7 = (counts['n_obs'] >= 7).sum()
    n_8 = (counts['n_obs'] >= 8).sum()
    n_9 = (counts['n_obs'] >= 9).sum()

    print(n_5, n_6, n_7, n_8, n_9)

    print(n_5 / total, n_6 / total, n_7 / total , n_8 / total, n_9 / total)

Possible points for 7 days
total:  7882
6803 6424 6088 5707 5284
0.8631058107079421 0.8150215681299162 0.772392793707181 0.7240548084242578 0.6703882263384928
Possible points for 14 days
total:  7872
6655 6234 5855 5381 4915
0.8454014227642277 0.791920731707317 0.743775406504065 0.6835619918699187 0.624364837398374
Possible points for 21 days
total:  7854
6417 6018 5531 4956 4366
0.8170359052711994 0.7662337662337663 0.704227145403616 0.6310160427807486 0.5558950853068501
Possible points for 30 days
total:  7785
6092 5526 4879 4275 3643
0.7825305073859987 0.7098265895953757 0.6267180475272961 0.5491329479768786 0.46795118818240206


In [10]:
#looking for 8 time points before 1 week
#choosing 8 time points (~72% of samples) - 5707 total
#1 week only
df_1week = pastdf[
        (pastdf["satellite_date"] - pastdf["sample_date"]).abs().dt.days >= 7
    ]
df_1week_8 = (
    df_1week.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 8)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(8)
)
print(df_1week_8.shape)
print(df_1week_8.shape[0] / 8)
counts = (
        df_1week_8
        .groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
        .size()
        .reset_index(name='n_obs')
    )
print(len(counts))

(45656, 18)
5707.0
5707


In [11]:
#looking for 7 time points before 2 weeks
#choosing 7 time points (~74% of samples) - 5855 total
#2 week only
df_2week = pastdf[
        (pastdf["satellite_date"] - pastdf["sample_date"]).abs().dt.days >= 14
    ]
df_2week_7 = (
    df_2week.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 7)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(7)
)
print(df_2week_7.shape)
print(df_2week_7.shape[0] / 7)
counts = (
        df_2week_7
        .groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
        .size()
        .reset_index(name='n_obs')
    )
print(len(counts))

(40985, 18)
5855.0
5855


In [12]:
#looking for 7 time points before 3 weeks (maybe 5)
#choosing 7 time points (~70% of samples) - 5531 total
#3 week only
df_3week = pastdf[
        (pastdf["satellite_date"] - pastdf["sample_date"]).abs().dt.days >= 21
    ]
df_3week_7 = (
    df_3week.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 7)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(7)
)
print(df_3week_7.shape)
print(df_3week_7.shape[0] / 7)
counts = (
        df_3week_7
        .groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
        .size()
        .reset_index(name='n_obs')
    )
print(len(counts))

(38717, 18)
5531.0
5531


In [13]:
#looking for time points before 1 month
#choosing 6 time points (~71% of the sample) - 5526
#1 month
df_1month = pastdf[
        (pastdf["satellite_date"] - pastdf["sample_date"]).abs().dt.days >= 30
    ]
df_1month_6 = (
    df_1month.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 6)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(6)
)
print(df_1month_6.shape)
print(df_1month_6.shape[0] / 6)
counts = (
        df_1month_6
        .groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
        .size()
        .reset_index(name='n_obs')
    )
print(len(counts))

(33156, 18)
5526.0
5526


In [14]:
#function to get calculated columns
def add_calc_cols(df):
    #calculating statistics
    group_cols = ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance']

    def compute_time_features(group):
        group = group.sort_values('satellite_date')
    
        #create list of differences within each sample
        diffs = group['satellite_date'].diff().dt.days.dropna()
    
        #calculating the total difference
        tot_diff = (group['sample_date'].iloc[0] - group['satellite_date'].iloc[0]).days
    
        return pd.Series({
            'mean_diff_days': diffs.mean(),
            'std_diff_days': diffs.std()
        })

    time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()
    return time_features

In [15]:
#1 week df
df_1week_time = add_calc_cols(df_1week_8)
df_1week_8 = df_1week_8.merge(df_1week_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(df_1week_8.shape)

(45656, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_59156/757883381.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [16]:
#2 weeks df
df_2week_time = add_calc_cols(df_2week_7)
df_2week_7 = df_2week_7.merge(df_2week_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(df_2week_7.shape)

(40985, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_59156/757883381.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [17]:
#3 weeks df
df_3week_time = add_calc_cols(df_3week_7)
df_3week_7 = df_3week_7.merge(df_3week_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(df_3week_7.shape)

(38717, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_59156/757883381.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [18]:
#1 month df
df_1month_time = add_calc_cols(df_1month_6)
df_1month_6 = df_1month_6.merge(df_1month_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(df_1month_6.shape)

(33156, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_59156/757883381.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [19]:
#making everything a csv and pushing to github
df_1week_8.to_csv('week1_8.csv', index = False)
df_2week_7.to_csv('week2_7.csv', index = False)
df_3week_7.to_csv('week3_7.csv', index = False)
df_1month_6.to_csv('month1_6.csv', index = False)